Target: churned (1 = no order in 180 days). Features: Week 2's engineered set — days_since_last_order,
order_velocity, avg_order_value, tenure_days, avg review stars. Business goal: flag at-risk customers early
enough for a Rs. 200 retention coupon to save a Rs. 4,000/year customer.

import libraries

In [38]:
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

load csv files

In [ ]:
orders = pd.read_csv("features.csv")
users = pd.read_csv("users.csv")

C:\Users\EMAN TAHIR\AppData\Local\Temp\ipykernel_1492\395137832.py:1: DtypeWarning: Columns (15,18) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv(r"C:\Users\EMAN TAHIR\Desktop\internship\ml handbook\ml\compiled features csv\features.csv")


convert dates

In [41]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
users["signup_date"] = pd.to_datetime(users["signup_date"])

merge csv files

In [42]:
data = orders.merge(users, on="user_id", how="left")

reference date for churn calculation

In [43]:
today = datetime(2026, 7, 23)

aggregate features per user

In [44]:
features = (
    data.groupby("user_id")
    .agg(last_order_date=("order_date", "max"), first_order_date=("order_date", "min"), order_count=("order_date", "count"), avg_order_value=("total_spend", "mean"), avg_review_stars=("total_orders", "mean")).reset_index())


engineer features from week 2

In [45]:
features["days_since_last_order"] = (today - features["last_order_date"]).dt.days
features["tenure_days"] = (today - features["first_order_date"]).dt.days
features["order_velocity"] = np.where( features["tenure_days"] > 0, features["order_count"] / features["tenure_days"], 0)

define churned target

In [46]:
features["churned"] = (features["days_since_last_order"] > 180).astype(int)
features.to_csv("churn_features.csv", index=False)
print("engineered features:\n", features.head())

engineered features:
    user_id last_order_date first_order_date  order_count  avg_order_value  \
0        0             NaT              NaT            0              NaN   
1        1             NaT              NaT            0              NaN   
2        2             NaT              NaT            0              NaN   
3        3             NaT              NaT            0              NaN   
4        4             NaT              NaT            0              NaN   

   avg_review_stars  days_since_last_order  tenure_days  order_velocity  \
0               NaN                    NaN          NaN             0.0   
1               NaN                    NaN          NaN             0.0   
2               NaN                    NaN          NaN             0.0   
3               NaN                    NaN          NaN             0.0   
4               NaN                    NaN          NaN             0.0   

   churned  
0        0  
1        0  
2        0  
3        0  

train churn model

In [47]:
X = features[["days_since_last_order", "order_velocity", "avg_order_value", "tenure_days", "avg_review_stars"]]
y = features["churned"]
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nModel Evaluation:\n", classification_report(y_test, y_pred))


Model Evaluation:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1399
           1       1.00      1.00      1.00       101

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500



flag at-risk customers


In [48]:
features["predicted_churn"] = model.predict(X)
features["coupon_offer"] = features["predicted_churn"].apply(lambda x: "Offer Rs.200 coupon" if x == 1 else "No coupon")
print("\nRetention Strategy:\n", features[["user_id", "predicted_churn", "coupon_offer"]].head())


Retention Strategy:
    user_id  predicted_churn coupon_offer
0        0                0    No coupon
1        1                0    No coupon
2        2                0    No coupon
3        3                0    No coupon
4        4                0    No coupon
